#  Excel File Cleaner & Consolidator
This notebook reads multiple Excel files from a folder, cleans each one, and consolidates them into a single output file.

**Workflow:**
1. Set your folder path and file list
2. Review the cleaning steps (customize as needed)
3. Run all cells — output saved as `consolidated_output.xlsx`

## 1. Import Libraries

In [ ]:
import pandas as pd
import os
from pathlib import Path

print(f"pandas version: {pd.__version__}")

## 2. Configuration — Set Your Paths Here

In [ ]:
# ─── CONFIGURE THESE ────────────────────────────────────────────────────────

# Folder where your Excel files are located (use '.' for same folder as notebook)
INPUT_FOLDER = "./data"  # <-- change this

# Output file name
OUTPUT_FILE = "consolidated_output.xlsx"

# Sheet name to read from each file (0 = first sheet, or use name e.g. 'Sheet1')
SHEET_NAME = 0

# Add a column to track which file each row came from?
ADD_SOURCE_COLUMN = True

# ────────────────────────────────────────────────────────────────────────────

# Auto-detect all .xlsx files in the folder
input_path = Path(INPUT_FOLDER)
excel_files = sorted(input_path.glob("*.xlsx"))

print(f"Found {len(excel_files)} Excel file(s):")
for f in excel_files:
    print(f"  - {f.name}")

## 3. Preview One File Before Processing

In [ ]:
if excel_files:
    sample = pd.read_excel(excel_files[0], sheet_name=SHEET_NAME)
    print(f"Shape: {sample.shape}")
    print(f"Columns: {list(sample.columns)}")
    print()
    display(sample.head())
else:
    print("No Excel files found. Check INPUT_FOLDER path.")

## 4. Cleaning Function — Customize Your Rules Here

In [ ]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply cleaning steps to a single DataFrame.
    Customize the steps inside this function to match your data.
    """
    # ── Step 1: Standardize column names ─────────────────────────────────────
    # Lowercase, strip spaces, replace spaces with underscores
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r'\s+', '_', regex=True)
        .str.replace(r'[^a-z0-9_]', '', regex=True)
    )

    # ── Step 2: Drop fully empty rows and columns ─────────────────────────────
    df = df.dropna(how='all')        # rows where ALL values are NaN
    df = df.dropna(axis=1, how='all') # columns where ALL values are NaN

    # ── Step 3: Strip whitespace from string columns ──────────────────────────
    str_cols = df.select_dtypes(include='object').columns
    df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())

    # ── Step 4: Reset index ───────────────────────────────────────────────────
    df = df.reset_index(drop=True)

    # ── ADD YOUR CUSTOM STEPS BELOW ───────────────────────────────────────────
    # Example: drop duplicate rows
    # df = df.drop_duplicates()

    # Example: fill NaN in a specific column
    # df['quantity'] = df['quantity'].fillna(0)

    # Example: convert a column to datetime
    # df['date'] = pd.to_datetime(df['date'], errors='coerce')

    # Example: rename a column
    # df = df.rename(columns={'old_name': 'new_name'})

    # Example: filter out rows based on a condition
    # df = df[df['status'] != 'cancelled']
    # ─────────────────────────────────────────────────────────────────────────

    return df

print(" clean_dataframe() function defined.")

## 5. Loop — Read, Clean, and Collect All Files

In [ ]:
all_frames = []
summary = []

for file in excel_files:
    print(f"Processing: {file.name}")
    
    try:
        # Read
        df = pd.read_excel(file, sheet_name=SHEET_NAME)
        rows_before = len(df)

        # Clean
        df = clean_dataframe(df)
        rows_after = len(df)

        # Optionally tag the source file
        if ADD_SOURCE_COLUMN:
            df.insert(0, 'source_file', file.name)

        all_frames.append(df)
        summary.append({
            'file': file.name,
            'rows_before': rows_before,
            'rows_after': rows_after,
            'rows_dropped': rows_before - rows_after,
            'columns': len(df.columns),
            'status': 'OK'
        })
        print(f"  → {rows_before} rows in | {rows_after} rows out | {rows_before - rows_after} dropped")

    except Exception as e:
        print(f"  ⚠️  ERROR: {e}")
        summary.append({'file': file.name, 'status': f'ERROR: {e}'})

print(f"\n✅ Done. {len(all_frames)}/{len(excel_files)} files processed successfully.")

## 6. Processing Summary

In [ ]:
summary_df = pd.DataFrame(summary)
display(summary_df)

## 7. Consolidate and Save Output

In [ ]:
if all_frames:
    consolidated = pd.concat(all_frames, ignore_index=True)

    print(f"Consolidated shape: {consolidated.shape}")
    print(f"Columns: {list(consolidated.columns)}")
    display(consolidated.head(10))

    # Save to Excel
    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
        consolidated.to_excel(writer, sheet_name='Consolidated', index=False)
        summary_df.to_excel(writer, sheet_name='Processing Summary', index=False)

    print(f"\n✅ Output saved to: {OUTPUT_FILE}")
else:
    print("No data to consolidate. Check errors above.")

## 8. Quick Data Quality Check on Output

In [ ]:
if all_frames:
    print("=== Data Types ===")
    print(consolidated.dtypes)
    print()
    print("=== Missing Values per Column ===")
    missing = consolidated.isnull().sum()
    missing_pct = (missing / len(consolidated) * 100).round(2)
    quality_df = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
    display(quality_df[quality_df['missing_count'] > 0])
    print()
    print("=== Basic Stats ===")
    display(consolidated.describe(include='all'))